In [0]:
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.sql.functions import col

model = LogisticRegressionModel.load("/Volumes/teleco_churn_datalakehouse/telecom_ml_database/models/my_model")

In [0]:
df = spark.table("teleco_churn_datalakehouse.gold.customer_churn_features")

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "Tenure_Months",
        "Monthly_Charges",
        "Total_Charges"
    ],
    outputCol="features",
    handleInvalid="skip"
)

df_features = assembler.transform(df)

In [0]:
predictions = model.transform(df_features)

In [0]:
predictions.select(
    "customer_key",
    "probability",
    "prediction"
).display()

In [0]:
# Save predictions as a table
from pyspark.sql.functions import udf
from pyspark.ml.linalg import VectorUDT, DenseVector
from pyspark.sql.types import DoubleType

# Minimal UDF to extract the second element of the probability vector
prob_udf = udf(lambda v: float(v[1]) if v is not None else None, DoubleType())

predictions_final = predictions.withColumn(
    "churn_probability",
    prob_udf("probability")
)

predictions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.telecom_ml_database.churn_predictions")

In [0]:
# Save predictions as a table
predictions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.telecom_ml_database.churn_predictions")

In [0]:
%sql
SELECT * 
FROM teleco_churn_datalakehouse.telecom_ml_database.churn_predictions
LIMIT 100 ;

In [0]:
# Defining high risk customers
high_risk_customers = predictions_final.filter(
    col("churn_probability") > 0.7
)

In [0]:
# Saving the high risk customers table
high_risk_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.telecom_ml_database.high_risk_customers")

In [0]:
%sql
SELECT * 
FROM teleco_churn_datalakehouse.telecom_ml_database.high_risk_customers
LIMIT 100 ;